# 🏛️ oracc-parser — Quickstart

This notebook shows you the actual data available, how to parse it, and where everything lives on disk.

## 🎯 Goals of this notebook

1. **Get the data** — download pre-processed CSVs and catalogues from Zenodo
2. **Parse a project** — load word CSVs, inspect the project catalogue, and explore individual tablets
3. **Get flat DataFrames** — metadata, transliterations, and full combined tables ready for analysis
4. **Export** — save results as CSV or JSONL

## 1. Get the data (Zenodo)

Download pre-processed word CSVs and project catalogues from Zenodo (doi: 10.5281/zenodo.19834993).
This is much faster than re-parsing the raw ORACC JSON files yourself.

In [1]:
from oracc_parser.download.fetch_data import fetch_data
from oracc_parser.settings import WORD_CSV_DIR, CATALOGUE_DIR, DATA_DIR, CACHE_DIR

# Data is ready if:
#   - catalogues/ has CSV files
#   - oracc_csvs.zip exists (not yet extracted) OR oracc_csvs/ already has project folders
#   - cache/html/ has translation files
catalogues_ok = CATALOGUE_DIR.exists() and any(CATALOGUE_DIR.glob("*.csv"))
csvs_ok = (DATA_DIR / "oracc_csvs.zip").exists() or (
    WORD_CSV_DIR.exists() and any(WORD_CSV_DIR.iterdir())
)
translations_ok = (CACHE_DIR / "html").exists() and any((CACHE_DIR / "html").rglob("*.html"))

if not (catalogues_ok and csvs_ok and translations_ok):
    print("Downloading pre-processed data from Zenodo (this may take a few minutes)...")
    fetch_data()
else:
    n_projects = sum(1 for d in WORD_CSV_DIR.iterdir() if d.is_dir()) if WORD_CSV_DIR.exists() else 0
    n_catalogues = len(list(CATALOGUE_DIR.glob("*.csv")))
    print(f"Data ready: {n_catalogues} catalogues, {n_projects} extracted project(s) in oracc_csvs/")

Data ready: 138 catalogues, 3 extracted project(s) in oracc_csvs/


## 2. Parse a project

Pick any project. We load its texts into a word-level dataframe, and inspect the project metadata.

#### Extract a project from the zip file and view its processed metadata

In [2]:
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_metadata_table
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR

# Change to any project available in oracc_csvs/
PROJECT = "saao/saa01"
LIMIT = 5  # Set to None to load all tablets

project_slug = PROJECT.replace("/", "-")
project_csv_dir = WORD_CSV_DIR / project_slug

# Loads from disk if already extracted; extracts from oracc_csvs.zip otherwise
word_dfs = load_word_csvs_from_dir(project_csv_dir, project=PROJECT)
if LIMIT:
    word_dfs = dict(list(word_dfs.items())[:LIMIT])

config = RunConfig()
records = parse_project_from_word_csvs(PROJECT, word_dfs, config=config)
print(f"Loaded {len(records)} tablets from {PROJECT}")

display(get_metadata_table(records))

12:54:29 | INFO    | oracc_parser | Loaded 264 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\saao-saa01
Processing saao/saa01: 100%|██████████| 5/5 [00:01<00:00,  3.51it/s]
12:54:30 | INFO    | oracc_parser | Processed 5 tablets for saao/saa01 from word CSVs


Loaded 5 tablets from saao/saa01


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,end_year
0,saao/saa01_P224395,saao/saa01,P224395,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,894019,Neo-Assyrian Empire,Neo-Assyrian,-721,-705
1,saao/saa01_P224403,saao/saa01,P224403,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,894019,Neo-Assyrian Empire,Neo-Assyrian,-717,-706
2,saao/saa01_P224417,saao/saa01,P224417,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,894019,Neo-Assyrian Empire,Neo-Assyrian,-721,-705
3,saao/saa01_P224431,saao/saa01,P224431,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,894019,Neo-Assyrian Empire,Neo-Assyrian,-721,-705
4,saao/saa01_P224433,saao/saa01,P224433,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,894019,Neo-Assyrian Empire,Neo-Assyrian,-721,-705


In [3]:
# Look at one tablet up close
tablet = records[0]

print(f"=== {tablet.metadata.identifier} ===")
print(f"   Project:     {PROJECT}")
print(f"   Text ID:     {tablet.metadata.id_text}")
print(f"   Genre:       {tablet.metadata.genre}")

geo = tablet.metadata.geographical_information
if geo:
    print(f"   Provenance:  {geo.city.city_name}")
    print(f"   Pleiades ID: {geo.city.city_plaides_id}")

chrono = tablet.metadata.chronological_information
if chrono:
    print(f"   Period:      {chrono.tablet_period.period_name if chrono.tablet_period else ''}")
    print(f"   Years:       {chrono.start_year} to {chrono.end_year}")

print(f"   Words:       {len(tablet.content.words)}")

=== saao/saa01_P224395 ===
   Project:     saao/saa01
   Text ID:     P224395
   Genre:       Administrative Letter
   Provenance:  Nimrud
   Pleiades ID: 894019
   Period:      Neo-Assyrian
   Years:       -721 to -705
   Words:       133


#### View one text

The texts were processed from the json format to a word-level DataFrame (see notebook 4 for more details on the process).

In [4]:
from oracc_parser.io.word_csv import record_to_word_dataframe

word_df = record_to_word_dataframe(records[0])
print(f"Word-level CSV for {records[0].metadata.id_text}: {len(word_df)} rows")
display(word_df.head(10))

Word-level CSV for P224395: 133 rows


,text_id,project,word_index,frag,ref,inst,form,lemma_form,sense,norm,raw_pos,lang,line,signs_reading,signs_break_pct,unicode,break_info
0,P224395,saao/saa01,0,a-na\t,P224395.2.1,ana[to]PRP,a-na,ana,to,ana,PRP,akk-x-neoass,2,a-na,0.00,𒀀; 𒈾,complete; complete
1,P224395,saao/saa01,1,⸢LUGAL⸣,P224395.2.2,šarri[king]N,LUGAL,šarru,king,šarri,N,akk-x-neoass,2,⸢LUGAL⸣,0.50,𒈗,damaged
2,P224395,saao/saa01,2,EN-⸢ia⸣,P224395.2.3,bēlīya[lord]N,EN-ia,bēlu,lord,bēlīya,N,akk-x-neoass,2,EN-⸢ia⸣,0.25,𒂗; 𒅀,complete; damaged
3,P224395,saao/saa01,3,ARAD-ka,P224395.3.1,urdaka[servant]N,ARAD-ka,ardu,servant,urdaka,N,akk-x-neoass,3,ARAD-ka,0.00,𒀴; 𒅗,complete; complete
4,P224395,saao/saa01,4,{1}10—⸢ha⸣-ti,P224395.3.2,Adda-hati[1]PN,{1}10-ha-ti,Adda-hati,1,Adda-hati,PN,akk-x-neoass,3,{1}10-⸢ha⸣-ti,0.12,𒁹; 𒌋; 𒄩; 𒋾,complete; complete; damaged; complete
5,P224395,saao/saa01,5,⸢lu⸣,P224395.4.1,lū[may]MOD,lu,lū,may,lū,MOD,akk-x-neoass,4,⸢lu⸣,0.50,𒇻,damaged
6,P224395,saao/saa01,6,DI-mu,P224395.4.2,šulmu[health]N,DI-mu,šulmu,health,šulmu,N,akk-x-neoass,4,DI-mu,0.00,𒁲; 𒈬,complete; complete
7,P224395,saao/saa01,7,⸢a-na\t,P224395.4.3,ana[to]PRP,a-na,ana,to,ana,PRP,akk-x-neoass,4,⸢a-na⸣,0.50,𒀀; 𒈾,damaged; damaged
8,P224395,saao/saa01,8,LUGAL,P224395.4.4,šarri[king]N,LUGAL,šarru,king,šarri,N,akk-x-neoass,4,⸢LUGAL⸣,0.50,𒈗,damaged
9,P224395,saao/saa01,9,EN⸣-ia,P224395.4.5,bēlīya[lord]N,EN-ia,bēlu,lord,bēlīya,N,akk-x-neoass,4,⸢EN⸣-ia,0.25,𒂗; 𒅀,damaged; complete


The word-level DataFrames can then be processed to represent the text in transliteration, normalization, Unicode cuneiform, or the translation

In [5]:
# The text in different representations
c = tablet.content

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:200] if c.transliterated_str_representation else '(none)'}")
print()
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:200] if c.normalized_str_representation else '(none)'}")
print()
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:200] if c.unicode_str_representation else '(none)'}")
print()
print("ENGLISH TRANSLATION:")
print(f"{c.english_translation[:200] if c.english_translation else '(none)'}")

TRANSLITERATION:
a-na\t ⸢LUGAL⸣ EN-⸢ia⸣
ARAD-ka {1}10—⸢ha⸣-ti
⸢lu⸣ DI-mu ⸢a-na\t LUGAL EN⸣-ia
⸢DUMU⸣ {1}⸢a⸣-mi-ri
ina ŠA₃-bi 03 me⸣ {anše}a-na-qa-te
u₂-za-ta-⸢ki⸣ ma-a ina UGU
{⸢lu₂v⸣}hu-ub-⸢te ša TA@v\m⸣
{⸢uru⸣}[di]-

NORMALIZATION:
ana šarri bēlīya
urdaka Adda-hati
lū šulmu ana šarri bēlīya
mār Amiri
ina libbi UNK mē anāqāte
uzzatakki mā ina muhhi
hubte ša issu
Dimašqa ana māt Aššur
ušettaqūni
mā ina muhhi amaqqut
assēme ina muh

UNICODE CUNEIFORM:
𒀀𒈾 𒈗 𒂗𒅀
𒀴𒅗 𒁹𒌋𒄩𒋾
𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀
𒌉 𒁹𒀀𒈪𒊑
𒀸 𒊮𒁉 𒐈 𒈨 𒀲𒀀𒈾𒋡𒋼
𒌑𒍝𒋫𒆠 𒈠𒀀 𒀸 𒌋𒅗
𒄷𒌒𒋼 𒊭 𒋬
𒌷𒁲𒈦𒋡 𒀀𒈾 𒆳𒀸𒋩 𒆳𒀸𒋩
𒌑𒊺𒋫𒄣𒌑𒉌
𒈠𒀀 𒀸 𒌋𒅗𒄭 𒀀𒈠𒄣𒌓
𒀀𒋛𒈨 𒀸 𒌋𒅗 𒁹𒂗𒅅𒁉
𒀀𒉺𒅁𒊏 𒄿𒊑𒅗
𒄿𒊓𒀀𒄭𒅆 𒀸 𒃮
𒄷𒌒𒋼 𒉌𒋫𒋃
𒂊𒋫𒈥𒈾𒀀𒅆
𒀸 𒊒𒌓 𒀸 𒆪𒋫𒇷𒉌
𒌑𒋛𒈨
𒉌𒀉𒋫𒄩𒍝 o
x𒅆𒐊𒈨 𒇻𒄭𒀀𒎌
𒄷𒌒𒋼 𒋬 𒌷

ENGLISH TRANSLATION:
To the king, my lord: Your servant Adda-hati. Good health to the king, my lord!
(Ammili'ti) the son of Amiri readied himself with 300 she-camels, intending to attack the booty being [tran]sferred from


## 3. Get flat DataFrames (for analysis & export)

Each function returns a clean pandas DataFrame — no nesting, no Pydantic.

In [6]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
display(flat)

Full table: 5 rows × 16 columns
Columns: ['id', 'project', 'text_id', 'genre', 'archive', 'provenance', 'period', 'start_year', 'end_year', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,saao/saa01_P224395,saao/saa01,P224395,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,a-na\t ⸢LUGAL⸣ EN-⸢ia⸣\nARAD-ka {1}10—⸢ha⸣-ti\...,ana šarri bēlīya\nurdaka Adda-hati\nlū šulmu a...,ana šarru bēlu\nardu Adda-hati\nlū šulmu ana š...,𒀀𒈾 𒈗 𒂗𒅀\n𒀴𒅗 𒁹𒌋𒄩𒋾\n𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒌉 𒁹𒀀𒈪𒊑\n𒀸 𒊮𒁉 𒐈...,"To the king, my lord: Your servant Adda-hati. ...",131,123
1,saao/saa01_P224403,saao/saa01,P224403,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-717,-706,a-bat LUGAL a-na\t {lu₂v}⸢šak⸣-[ni]\n07 me {tu...,abat šarri ana šakni\nUNK mē maqarrāt ša tibni...,awātu šarru ana šaknu\nUNK meʾatu maqarratu ša...,𒀀𒁁 𒈗 𒀀𒈾 𒊕𒉌\n𒐌 𒈨 𒌆𒈠𒃼𒋥 𒊭 𒊺𒅔𒉡\n𒐌 𒈨 𒂊𒁉𒄑𒋢 𒊭 𒄀𒆹𒎌\n𒊭...,The king's word to the go[vernor] (of Calah):\...,34,34
2,saao/saa01_P224417,saao/saa01,P224417,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,a-na\t LUGAL EN-ia\nARAD\d-ka {1}10—ha-ti\t\n⸢...,ana šarri bēlīya\nurdaka Adda-hati\nlū šulmu a...,ana šarru bēlu\nardu Adda-hati\nlū šulmu ana š...,𒀀𒈾 𒈗 𒂗𒅀\n𒀴𒅗 𒁹𒌋𒄩𒋾\n𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒆬𒌓 𒊭 𒃻𒉡𒈨𒋼 𒊭 ...,"To the king, my lord: Your servant Adda-hati. ...",193,177
3,saao/saa01_P224431,saao/saa01,P224431,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,a-na\t [LUGAL EN-ia₂]\nARAD-⸢ka⸣ {1}EN—BAD₃]\n...,ana šarri bēlīya\nurdaka Bel-duri\nšarru bēlī ...,ana šarru bēlu\nardu Bel-duri\nšarru bēlu ṭēmu...,𒀀𒈾 𒈗 𒂗𒐊\n𒀴𒅗 𒁹𒂗𒂦\n𒈗 𒁁𒉌 𒉈𒈬 𒄑𒊓𒃶\n𒈠𒀀 𒂗𒉆𒎌 𒃮𒁍\n𒃻𒈨 𒆠...,"To [the king, my lord: yo]ur servant [Bel-duri...",146,143
4,saao/saa01_P224433,saao/saa01,P224433,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,[a-na LUGAL EN-ia ARAD]-⸢ka⸣ {1}{d}30—PAB-MEŠ—...,ana šarri bēlīya urdaka Sin-ahhe-eriba\nlū šul...,ana šarru bēlu ardu Sin-ahhe-eriba\nlū šulmu a...,𒀀𒈾 𒈗 𒂗𒅀 𒀴𒅗 𒁹𒀭𒌍𒉽𒎌𒋢\n𒇻 𒂄𒈬 𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒂄𒈬 𒀀𒈾 𒆳𒀸𒋩𒆠\n...,"[To the king, my lord: yo]ur [servant S[in-ahh...",304,184


## 4. Export & see where output goes

In [7]:
from oracc_parser.settings import OUTPUT_DIR

out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path   = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"Exported to:")
print(f"  JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"  CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print(f"\nOutput directory: {out_dir}")

Exported to:
  JSONL: G:\My Drive\GitHub\oracc-parser\enriched_data\output\saao_saa01.jsonl  (28.5 KB)
  CSV:   G:\My Drive\GitHub\oracc-parser\enriched_data\output\saao_saa01.csv  (26.8 KB)

Output directory: G:\My Drive\GitHub\oracc-parser\enriched_data\output


## What's next?

- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **Configure parsing:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.